In [ ]:
"""
Simple Text-to-Image Generator
Uses primary model, automatically falls back to alternatives only if needed

INSTALLATION:
    pip install huggingface-hub pillow
"""
from huggingface_hub import InferenceClient
from datetime import datetime
from PIL import Image
from config import HF_API_KEY

# MODEL PRIORITY LIST - Primary model first, fallbacks only if it fails
MODELS = [# it has many models
    "ByteDance/SDXL-Lightning",
    "stabilityai/stable-diffusion-xl-base-1.0",
    "stabilityai/sdxl-turbo",
    "runwayml/stable-diffusion-v1-5", # Fallback 2
]

# Initialize client
client = InferenceClient(api_key=HF_API_KEY)

print(f"Primary model: {MODELS[0]}")
print("Type 'quit' to exit\n")

while True:#
    prompt = input("Enter prompt: ").strip()
    if prompt.lower() in ["quit", "exit", "q"]:
        break
    if not prompt:
        continue

    print("Generating...")
    image = None

    # Try each model in order until one works
    for model in MODELS:
        try:
            image = client.text_to_image(prompt, model=model)
            break  # Success! Exit the loop
        except Exception:
            print(f"  Executing next...")
            continue

    # If we got an image, save and display it
    if image:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"generated_{timestamp}.png"
        image.save(filename)
        print(f"✓ Saved: {filename}")
        image.show()
        print()
    else:
        print("Error: All models failed. Check your API key.\n")

print("Goodbye!")


In [ ]:
# pip install requests

import os
import time
import requests
from typing import Optional, Dict, Any

# =========================================================
# CONFIG
# =========================================================
HF_TOKEN = os.getenv("HF_TOKEN", "YOUR_HF_TOKEN")

# Pick a text-to-image model available on Hugging Face
MODEL = "black-forest-labs/FLUX.1-dev"

# Hugging Face router/inference endpoint pattern for model inference
API_URL = f"https://router.huggingface.co/hf-inference/models/{MODEL}"

HEADERS = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json",
}

# =========================================================
# PAYLOAD BUILDER
# =========================================================
def build_payload(
    prompt: str,
    negative_prompt: Optional[str] = None,
    width: int = 1024,
    height: int = 1024,
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
    seed: Optional[int] = 42,
    num_images: int = 1,
    extra_parameters: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Build a customizable text-to-image payload.

    Fields like num_inference_steps, guidance_scale, seed, and image size
    are common generation controls for diffusion-style image models.
    """
    parameters = {
        "width": width,
        "height": height,
        "num_inference_steps": num_inference_steps,
        "guidance_scale": guidance_scale,
        "num_images": num_images,
    }

    if negative_prompt:
        parameters["negative_prompt"] = negative_prompt

    if seed is not None:
        parameters["seed"] = seed

    if extra_parameters:
        parameters.update(extra_parameters)

    payload = {
        "inputs": prompt,
        "parameters": parameters,
        "options": {
            "wait_for_model": True,
            "use_cache": False
        }
    }
    return payload

# =========================================================
# GENERATION FUNCTION
# =========================================================
def generate_image(
    prompt: str,
    output_path: str = "generated_image.png",
    negative_prompt: Optional[str] = None,
    width: int = 1024,
    height: int = 1024,
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
    seed: Optional[int] = 42,
    num_images: int = 1,
    extra_parameters: Optional[Dict[str, Any]] = None,
    timeout: int = 180,
) -> None:
    payload = build_payload(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        seed=seed,
        num_images=num_images,
        extra_parameters=extra_parameters,
    )

    print("Sending payload:")
    print(payload)

    response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=timeout)

    # Handle common API issues
    if response.status_code == 200:
        content_type = response.headers.get("content-type", "")
        if "image" in content_type:
            with open(output_path, "wb") as f:
                f.write(response.content)
            print(f"Image saved as: {output_path}")
        else:
            print("Received non-image response:")
            print(response.text)

    elif response.status_code in (503, 504):
        print("Model may still be loading. Try again in a few moments.")
        print(response.text)

    else:
        print(f"Error: {response.status_code}")
        print(response.text)

# =========================================================
# EXAMPLE USAGE
# =========================================================
if __name__ == "__main__":
    prompt = (
        "A cinematic futuristic city at sunset, ultra-detailed, glass buildings, "
        "soft volumetric lighting, premium sci-fi aesthetic, highly realistic"
    )

    negative_prompt = (
        "blurry, low quality, distorted face, extra fingers, watermark, text, logo"
    )

    # Extra optional controls can be added here if your selected model/provider supports them
    extra = {
        # "scheduler": "DPMSolverMultistep",
        # "style_preset": "cinematic",
    }

    generate_image(
        prompt=prompt,
        output_path="city.png",
        negative_prompt=negative_prompt,
        width=1024,
        height=1024,
        num_inference_steps=35,
        guidance_scale=8.0,
        seed=1234,
        num_images=1,
        extra_parameters=extra,
    )

In [ ]:
"""
Simple Text-to-Image Generator
Uses primary model, automatically falls back to alternatives only if needed

INSTALLATION:
    pip install huggingface-hub pillow
"""
from huggingface_hub import InferenceClient
from datetime import datetime
from PIL import Image
from config import HF_API_KEY

# MODEL PRIORITY LIST - Primary model first, fallbacks only if it fails
MODELS = [
    "ByteDance/SDXL-Lightning",
    "stabilityai/stable-diffusion-xl-base-1.0",
    "stabilityai/sdxl-turbo",
    "runwayml/stable-diffusion-v1-5", # Fallback 2
]

# Initialize client
client = InferenceClient(api_key=HF_API_KEY)

print(f"Primary model: {MODELS[0]}")
print("Type 'quit' to exit\n")

while True:
    prompt = input("Enter prompt: ").strip()
    if prompt.lower() in ["quit", "exit", "q"]:
        break
    if not prompt:
        continue

    print("Generating...")
    image = None

    # Try each model in order until one works
    for model in MODELS:
        try:
            image = client.text_to_image(prompt, model=model)
            break  # Success! Exit the loop
        except Exception:
            print(f"  Executing next...")
            continue

    # If we got an image, save and display it
    if image:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"generated_{timestamp}.png"
        image.save(filename)
        print(f"✓ Saved: {filename}")
        image.show()
        print()
    else:
        print("Error: All models failed. Check your API key.\n")

print("Goodbye!")


In [ ]:
from openai import OpenAI
import base64
import json
from pathlib import Path

client = OpenAI()

def build_stylized_prompt(
    subject,
    style=None,
    lighting=None,
    color_palette=None,
    camera=None,
    mood=None,
    extra_details=None
):
    """
    Build a structured, layered prompt for stylistic nuance.
    """

    components = [f"Subject: {subject}"]

    if style:
        components.append(f"Art Style: {style}")

    if lighting:
        components.append(f"Lighting: {lighting}")

    if color_palette:
        components.append(f"Color Palette: {color_palette}")

    if camera:
        components.append(f"Camera: {camera}")

    if mood:
        components.append(f"Mood: {mood}")

    if extra_details:
        components.append(f"Additional Details: {extra_details}")

    return "\n".join(components)


def generate_image(
    prompt,
    negative_prompt=None,
    size="1024x1024",
    quality="high",
    style_strength=0.8,
    seed=None,
    background="auto",
    output_format="png",
    moderation="auto"
):
    """
    Generate an image with advanced configuration controls.
    """

    payload = {
        "model": "gpt-image-1",
        "prompt": prompt,
        "size": size,
        "quality": quality,
        "background": background,
        "output_format": output_format,
    }

    # Optional nuanced controls
    if negative_prompt:
        payload["negative_prompt"] = negative_prompt

    if seed is not None:
        payload["seed"] = seed  # For reproducibility

    # Some APIs support style strength / guidance scale
    payload["style_strength"] = style_strength

    # Optional moderation tuning
    payload["moderation"] = moderation

    print("Final Payload:\n", json.dumps(payload, indent=2))

    result = client.images.generate(**payload)

    image_base64 = result.data[0].b64_json
    image_bytes = base64.b64decode(image_base64)

    file_path = Path(f"generated_image.{output_format}")
    file_path.write_bytes(image_bytes)

    print(f"Image saved to {file_path.resolve()}")


# -----------------------------
# Example Usage
# -----------------------------

prompt = build_stylized_prompt(
    subject="A futuristic city skyline at sunset",
    style="cinematic sci-fi realism inspired by Blade Runner aesthetics",
    lighting="volumetric lighting, neon glow, soft haze",
    color_palette="deep blues, magenta highlights, high contrast",
    camera="wide-angle lens, shallow depth of field",
    mood="melancholic yet hopeful",
    extra_details="ultra-detailed textures, reflections on wet pavement, 8k resolution"
)

generate_image(
    prompt=prompt,
    negative_prompt="blurry, low resolution, distorted faces, oversaturated",
    size="1024x1024",
    quality="high",
    style_strength=0.9,
    seed=42,
    background="auto",
    output_format="png"
)